# **Bước 0: Thiết lập Môi trường và Tải Dữ liệu**

In [14]:
import pandas as pd
# Dữ liệu có thể được phân tách bằng tab và không có header
df_train = pd.read_csv('../data/hwu/train.csv', header=None, names=['text', 'intent'])
df_val   = pd.read_csv('../data/hwu/val.csv', header=None, names=['text', 'intent'])
df_test  = pd.read_csv('../data/hwu/test.csv', header=None, names=['text', 'intent'])
print("Train shape:", df_train.shape)
print("Validation shape:", df_val.shape)
print("Test shape:", df_test.shape)
df_train.head()

Train shape: (8955, 2)
Validation shape: (1077, 2)
Test shape: (1077, 2)


,text,intent
0,text,category
1,what alarms do i have set right now,alarm_query
2,checkout today alarm of meeting,alarm_query
3,report alarm settings,alarm_query
4,see see for me the alarms that you have set to...,alarm_query


In [15]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(pd.concat([df_train['intent'], df_val['intent'], df_test['intent']]))

y_train = label_encoder.transform(df_train['intent'])
y_val   = label_encoder.transform(df_val['intent'])
y_test  = label_encoder.transform(df_test['intent'])

num_classes = len(label_encoder.classes_)
num_classes

65

# **Task 1: (Warm-up Ôn bài cũ) Pipeline TF-IDF + Logistic Regression**

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, f1_score, log_loss

tfidf_lr = make_pipeline(
    TfidfVectorizer(max_features=5000),
    LogisticRegression(max_iter=1000)
)

tfidf_lr.fit(df_train['text'], y_train)

y_pred_lr = tfidf_lr.predict(df_test['text'])

# Predict probability để tính loss
y_pred_lr_prob = tfidf_lr.predict_proba(df_test['text'])

f1_lr = f1_score(y_test, y_pred_lr, average='macro')

print("\n===== TF-IDF + LOGISTIC REGRESSION RESULTS =====")
print("Macro F1-score:", f1_lr)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))



===== TF-IDF + LOGISTIC REGRESSION RESULTS =====
Macro F1-score: 0.8225667213437197

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.92        19
           1       1.00      0.73      0.84        11
           2       0.81      0.89      0.85        19
           3       1.00      0.75      0.86         8
           4       0.92      0.80      0.86        15
           5       0.93      1.00      0.96        13
           6       0.48      0.53      0.50        19
           7       0.89      0.89      0.89        19
           8       0.82      0.74      0.78        19
           9       0.00      0.00      0.00         1
          10       0.59      0.68      0.63        19
          11       0.67      0.75      0.71         8
          12       0.74      0.89      0.81        19
          13       0.78      0.88      0.82         8
          14       0.83      0.79      0.81        19
          15       0.92   

c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

# **Task 2: Word2Vec trung bình + Dense Layer**

In [53]:
import numpy as np
import tensorflow as tf
import random
from gensim.models import Word2Vec
from sklearn.metrics import f1_score, classification_report

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)


# 1. LABEL ENCODING

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(pd.concat([df_train['intent'], df_val['intent'], df_test['intent']]))

y_train = label_encoder.transform(df_train['intent'])
y_val   = label_encoder.transform(df_val['intent'])
y_test  = label_encoder.transform(df_test['intent'])

num_classes = len(label_encoder.classes_)
print("Số lớp:", num_classes)


# 2. TRAIN WORD2VEC

sentences = [t.split() for t in df_train['text']]

w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,     # tránh từ rác
    workers=4,
    epochs=50        # embedding tốt hơn
)

# 3. CÂU  VECTOR TRUNG BÌNH
def sentence_to_avg(text, model):
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

X_train_avg = np.array([sentence_to_avg(t, w2v) for t in df_train['text']])
X_val_avg   = np.array([sentence_to_avg(t, w2v) for t in df_val['text']])
X_test_avg  = np.array([sentence_to_avg(t, w2v) for t in df_test['text']])

# 4. MODEL DENSE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model_avg = Sequential([
    Dense(128, activation='relu', input_shape=(w2v.vector_size,)),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model_avg.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5. TRAIN

model_avg.fit(
    X_train_avg, y_train,
    validation_data=(X_val_avg, y_val),
    epochs=20,
    batch_size=16
)

Số lớp: 65
Epoch 1/20


c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.5178 - loss: 2.0297 - val_accuracy: 0.7298 - val_loss: 1.0907
Epoch 2/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7089 - loss: 1.0540 - val_accuracy: 0.7502 - val_loss: 0.8852
Epoch 3/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7448 - loss: 0.9055 - val_accuracy: 0.7697 - val_loss: 0.8097
Epoch 4/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7627 - loss: 0.8314 - val_accuracy: 0.7799 - val_loss: 0.7635
Epoch 5/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7819 - loss: 0.7694 - val_accuracy: 0.7864 - val_loss: 0.7369
Epoch 6/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7868 - loss: 0.7314 - val_accuracy: 0.7920 - val_loss: 0.7154
Epoch 7/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7951 - loss: 0.7020 - val_accuracy: 0.7985 - val_loss: 0.6977
Epoch 8/20
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8042 - loss: 0.6764 - val_accuracy: 0.8032 - val_

In [54]:
from sklearn.metrics import classification_report, f1_score

# Evaluation
y_pred_prob = model_avg.predict(X_test_avg)
y_pred = np.argmax(y_pred_prob, axis=1)

f1_macro = f1_score(y_test, y_pred, average='macro')
test_loss = model_avg.evaluate(X_test_avg, y_test, verbose=0)[0]

print("\n===== WORD2VEC (AVG) + DENSE RESULTS =====")
print("Macro F1-score:", f1_macro)
print("Test Loss:", test_loss)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 

===== WORD2VEC (AVG) + DENSE RESULTS =====
Macro F1-score: 0.7968963039739305
Test Loss: 0.7247587442398071

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.89      0.92        19
           1       0.82      0.82      0.82        11
           2       0.85      0.89      0.87        19
           3       0.47      0.88      0.61         8
           4       0.71      0.80      0.75        15
           5       0.89      0.62      0.73        13
           6       0.41      0.47      0.44        19
           7       1.00      0.89      0.94        19
           8       0.85      0.58      0.69        19
           9       1.00      1.00      1.00         1
          10       0.75      0.47      0.58        19
          11       0.75      0.75      0.75         8
          12       0.79      0.79      0.79        19
          13       0.80      1.00      0.89         8
          14    

# **Task 3: LSTM + Pre-trained Word2Vec Embeddings**

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from gensim.models import Word2Vec

tokenizer = Tokenizer(oov_token="<UNK>")
tokenizer.fit_on_texts(df_train['text'])

X_train_seq = tokenizer.texts_to_sequences(df_train['text'])
X_val_seq   = tokenizer.texts_to_sequences(df_val['text'])
X_test_seq  = tokenizer.texts_to_sequences(df_test['text'])

max_len = 60
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=max_len, padding='post')

vocab = tokenizer.word_index
vocab_size = len(vocab) + 1

# 2. WORD2VEC TRAINING (token-based, chuẩn nhất)
sentences = [t.split() for t in df_train['text']]

w2v = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=50
)

embedding_dim = w2v.vector_size

# 3. BUILD EMBEDDING MATRIX
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in vocab.items():
    if word in w2v.wv:
        embedding_matrix[idx] = w2v.wv[word]

# 4. MODEL: LSTM + Word2Vec PRETRAINED
model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=True        #  cho fine-tune => F1 tăng mạnh
    ),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2, return_sequences=False),
    BatchNormalization(),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
# 5. TRAIN
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)




Epoch 1/20


c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 32ms/step - accuracy: 0.0194 - loss: 4.1472 - val_accuracy: 0.0399 - val_loss: 4.0737
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.0383 - loss: 3.9889 - val_accuracy: 0.0631 - val_loss: 3.7528
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.0664 - loss: 3.6379 - val_accuracy: 0.1170 - val_loss: 3.2566
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 33ms/step - accuracy: 0.1002 - loss: 3.2633 - val_accuracy: 0.1458 - val_loss: 2.9241
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 9s 31ms/step - accuracy: 0.1273 - loss: 3.0513 - val_accuracy: 0.1588 - val_loss: 2.7814
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.1680 - loss: 2.8149 - val_accuracy: 0.2284 - val_loss: 2.5169
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.2147 - loss: 2.6229 - val_accuracy: 0.2878 - val_loss: 2.4161
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 8s 29ms/step - accuracy: 0.2590 - loss: 2.4043 - val_accuracy: 0.2

c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [57]:
# 6. EVALUATION
y_pred_prob = model.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

f1_pretrained = f1_score(y_test, y_pred, average='macro')
loss_pretrained = log_loss(y_test, y_pred_prob)

print("\n===== LSTM + Word2Vec PRETRAINED (Optimized) =====")
print("Macro F1-score:", f1_pretrained)
print("Test Loss:", loss_pretrained)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step

===== LSTM + Word2Vec PRETRAINED (Optimized) =====
Macro F1-score: 0.6408674993835819
Test Loss: 1.0504123011430198

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.95      0.90        19
           1       0.82      0.82      0.82        11
           2       0.81      0.89      0.85        19
           3       0.00      0.00      0.00         8
           4       0.44      0.27      0.33        15
           5       0.41      0.69      0.51        13
           6       0.40      0.42      0.41        19
           7       0.80      0.84      0.82        19
           8       0.86      0.63      0.73        19
           9       0.00      0.00      0.00         1
          10       0.79      0.79      0.79        19
          11       0.64      0.88      0.74         8
          12       0.70      0.84      0.76        19
          13       1.00      0.50      0.67         8
        

c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

# **Task 4: Mô hình Nâng cao (Embedding học từ đầu + LSTM)**

In [56]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import f1_score, classification_report, log_loss

# MODEL LSTM SCRATCH (tối ưu)
lstm_scratch = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=200,         # embedding lớn hơn => F1 tăng
        input_length=max_len
    ),
    
    LSTM(
        128,
        dropout=0.3,
        recurrent_dropout=0.2,
        return_sequences=False
    ),

    BatchNormalization(),      #  giúp ổn định và tăng F1

    Dense(64, activation='relu'),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

lstm_scratch.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
history_scratch = lstm_scratch.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)



Epoch 1/20


c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


280/280 ━━━━━━━━━━━━━━━━━━━━ 17s 51ms/step - accuracy: 0.0183 - loss: 4.1877 - val_accuracy: 0.0176 - val_loss: 4.1537
Epoch 2/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.0172 - loss: 4.1572 - val_accuracy: 0.0176 - val_loss: 4.1438
Epoch 3/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 43ms/step - accuracy: 0.0185 - loss: 4.1499 - val_accuracy: 0.0176 - val_loss: 4.1457
Epoch 4/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.0143 - loss: 4.1462 - val_accuracy: 0.0176 - val_loss: 4.1344
Epoch 5/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 47ms/step - accuracy: 0.0154 - loss: 4.1441 - val_accuracy: 0.0176 - val_loss: 4.1348
Epoch 6/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 43ms/step - accuracy: 0.0179 - loss: 4.1430 - val_accuracy: 0.0176 - val_loss: 4.1425
Epoch 7/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 13s 45ms/step - accuracy: 0.0185 - loss: 4.1394 - val_accuracy: 0.0176 - val_loss: 4.1318
Epoch 8/20
280/280 ━━━━━━━━━━━━━━━━━━━━ 12s 42ms/step - accuracy: 0.0149 - loss: 4.1391 - val_accurac

c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

In [58]:
# EVALUATION
y_pred_prob = lstm_scratch.predict(X_test_pad)
y_pred = np.argmax(y_pred_prob, axis=1)

f1_scratch = f1_score(y_test, y_pred, average='macro')
loss_scratch = log_loss(y_test, y_pred_prob)

print("\n===== LSTM (TRAINED FROM SCRATCH) =====")
print("Macro F1-score:", f1_scratch)
print("Test Loss:", loss_scratch)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

===== LSTM (TRAINED FROM SCRATCH) =====
Macro F1-score: 0.0005334081976417742
Test Loss: 4.128991572939181

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        19
           1       0.00      0.00      0.00        11
           2       0.00      0.00      0.00        19
           3       0.00      0.00      0.00         8
           4       0.00      0.00      0.00        15
           5       0.00      0.00      0.00        13
           6       0.00      0.00      0.00        19
           7       0.00      0.00      0.00        19
           8       0.00      0.00      0.00        19
           9       0.00      0.00      0.00         1
          10       0.00      0.00      0.00        19
          11       0.00      0.00      0.00         8
          12       0.00      0.00      0.00        19
          13       0.00      0.00      0.00         8
          14     

c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\DELL\Downloads\NLP_DL\nlp-labs\nlp-labs\nlp_dl\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

# **Ták 5: Đánh giá, So sánh và Phân tích**

In [64]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "TF-IDF + Logistic Regression",
        "Word2Vec Avg + Dense",
        "LSTM + Pretrained Embedding",
        "LSTM + Scratch Embedding"
    ],
    "F1-macro": [
        f1_lr,
        f1_avg,
        f1_pretrained,
        f1_lstm_scratch
    ],
    "Test Loss": [
        loss_lr,
        loss_avg,
        loss_lstm_pre,
        loss_lstm_scratch
    ]
})

print("\n===== SUMMARY RESULTS =====")
print(results)



===== SUMMARY RESULTS =====
                          Model  F1-macro  Test Loss
0  TF-IDF + Logistic Regression  0.822567   1.052858
1          Word2Vec Avg + Dense  0.790718   0.717341
2   LSTM + Pretrained Embedding  0.640867   3.369634
3      LSTM + Scratch Embedding  0.000533   4.136817


In [65]:
# PHÂN TÍCH ĐỊNH TÍNH

test_sentences = [
    "can you remind me to not call my mom",              
    "is it going to be sunny or rainy tomorrow",        
    "find a flight from new york to london but not through paris"
]

# Gán nhãn thật DƯỚI DẠNG TEXT (KHÔNG transform)
true_labels_text = [
    "reminder_create",
    "weather_query",
    "flight_search"
]

# 1) TF-IDF + LR
pred_lr = tfidf_lr.predict(test_sentences)

# 2) Word2Vec Avg + Dense
X_avg = np.array([sentence_to_avg(t, w2v) for t in test_sentences])
pred_avg = np.argmax(model_avg.predict(X_avg), axis=1)
pred_avg = label_encoder.inverse_transform(pred_avg)

# 3) LSTM pretrained
seqs = tokenizer.texts_to_sequences(test_sentences)
pads = pad_sequences(seqs, maxlen=max_len, padding='post')
pred_pre = np.argmax(lstm_pretrained.predict(pads), axis=1)
pred_pre = label_encoder.inverse_transform(pred_pre)

# 4) LSTM scratch
pred_scr = np.argmax(lstm_scratch.predict(pads), axis=1)
pred_scr = label_encoder.inverse_transform(pred_scr)

# Đưa tất cả vào bảng so sánh
analysis = pd.DataFrame({
    "Sentence": test_sentences,
    "True Intent": true_labels_text,
    "TF-IDF + LR": label_encoder.inverse_transform(pred_lr),
    "W2V Avg + Dense": pred_avg,
    "LSTM Pretrained": pred_pre,
    "LSTM Scratch": pred_scr,
})

print("\n===== QUALITATIVE ANALYSIS =====")
print(analysis)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 215ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

===== QUALITATIVE ANALYSIS =====
                                            Sentence      True Intent  \
0               can you remind me to not call my mom  reminder_create   
1          is it going to be sunny or rainy tomorrow    weather_query   
2  find a flight from new york to london but not ...    flight_search   

      TF-IDF + LR W2V Avg + Dense LSTM Pretrained  LSTM Scratch  
0    calendar_set    calendar_set     qa_currency  lists_remove  
1   weather_query   weather_query     qa_currency  lists_remove  
2  general_negate     email_query  takeaway_order  lists_remove  
